# 🚴 Scraping - BikeclubPedalazo v6

**Base:** v3 con dos fixes críticos:  
- ✅ **Fix imágenes:** las imágenes son `data:image/jpeg;base64,...` embebidas en el HTML; ahora se extraen correctamente (ya no se descartan)
- ✅ **Fix scroll / productos:** el sitio no usa scroll-infinito real; se agrega detección de botón 'Cargar más' y fallback a paginación
- ✅ Scroll completo hasta que no aparezcan productos nuevos
- ✅ Imágenes: lee `data-src` antes que `src` (evita placeholders)
- ✅ Imágenes siempre con URL absoluta
- ✅ Procesamiento de 5 en 5 con log detallado
- ✅ Reintentos automáticos por producto fallido
- ✅ Reporte final: OK vs errores


In [ ]:
# ── CELDA 1: Instalar Chrome + chromedriver ──────────────────────────────────
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y ./google-chrome-stable_current_amd64.deb -q

# Instalar chromedriver que coincida con la versión de Chrome instalada
import subprocess, re

# Obtener versión de Chrome
result = subprocess.run(['/usr/bin/google-chrome', '--version'],
                        capture_output=True, text=True)
version_str = result.stdout.strip()
print(f'Chrome instalado: {version_str}')

# Extraer major version
match = re.search(r'(\d+)\.', version_str)
major = match.group(1) if match else None
print(f'Major version: {major}')

# Instalar chromedriver del sistema vía apt (misma versión que Chrome)
!apt-get install -y chromium-driver -q 2>/dev/null || echo 'chromium-driver no disponible vía apt'

# Verificar si chromedriver quedó en el sistema
import shutil
cd_path = shutil.which('chromedriver')
if cd_path:
    r2 = subprocess.run([cd_path, '--version'], capture_output=True, text=True)
    print(f'chromedriver del sistema: {r2.stdout.strip()}')
else:
    print('chromedriver no encontrado en PATH, se usará WebDriverManager como fallback')

!pip install selenium webdriver-manager beautifulsoup4 pandas lxml -q
print('✅ Listo')


In [ ]:
# ── CELDA 2: Imports ────────────────────────────────────────────────────────
import time
import math
import re
from urllib.parse import urljoin

import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

print('✅ Imports OK')

In [ ]:
# ── CELDA 3: Configuración ──────────────────────────────────────────────────
URL_BASE = 'https://www.bikeclubpedalazo.online/productos.html'
DOMINIO  = 'https://www.bikeclubpedalazo.online'

BATCH_SIZE          = 5    # procesar de 5 en 5
MAX_REINTENTOS      = 3    # reintentos por producto fallido
SCROLL_PAUSA        = 3    # segundos de espera entre scrolls
SCROLL_MAX_SIN_CAMBIO = 4  # scrolls sin productos nuevos → detener

print('✅ Config OK')

In [ ]:
# ── CELDA 4: Driver con Google Chrome ──────────────────────────────────────
import subprocess, shutil, os

def crear_driver():
    opts = Options()
    opts.add_argument('--headless=new')
    opts.add_argument('--no-sandbox')
    opts.add_argument('--disable-dev-shm-usage')
    opts.add_argument('--disable-gpu')
    opts.add_argument('--window-size=1280,900')
    opts.add_argument('--disable-extensions')
    opts.add_argument('--disable-software-rasterizer')
    opts.add_argument('--remote-debugging-port=0')  # puerto aleatorio, evita conflictos
    opts.binary_location = '/usr/bin/google-chrome'

    # ── Estrategia 1: chromedriver del sistema (más rápido, sin descarga) ──
    for ruta in ['/usr/bin/chromedriver', '/usr/local/bin/chromedriver',
                 shutil.which('chromedriver') or '']:
        if ruta and os.path.exists(ruta):
            try:
                print(f'  🔧 Usando chromedriver del sistema: {ruta}')
                svc = Service(ruta)
                d = webdriver.Chrome(service=svc, options=opts)
                d.set_page_load_timeout(60)
                d.set_script_timeout(30)
                return d
            except Exception as e:
                print(f'  ⚠️ Falló {ruta}: {e}')

    # ── Estrategia 2: WebDriverManager con timeout extendido ────────────────
    try:
        print('  🔧 Descargando chromedriver con WebDriverManager...')
        from webdriver_manager.chrome import ChromeDriverManager
        from webdriver_manager.core.os_manager import ChromeType
        svc = Service(ChromeDriverManager().install())
        d = webdriver.Chrome(service=svc, options=opts)
        d.set_page_load_timeout(60)
        d.set_script_timeout(30)
        return d
    except Exception as e:
        print(f'  ⚠️ WebDriverManager falló: {e}')

    # ── Estrategia 3: selenium-manager (incluido en Selenium 4.6+) ──────────
    try:
        print('  🔧 Intentando con selenium-manager automático...')
        d = webdriver.Chrome(options=opts)  # selenium-manager resuelve solo
        d.set_page_load_timeout(60)
        d.set_script_timeout(30)
        return d
    except Exception as e:
        raise RuntimeError(f'No se pudo iniciar Chrome. Último error: {e}')

# Test rápido
print('🔄 Probando driver...')
d = crear_driver()
print('✅ Chrome headless iniciado correctamente')
d.quit()


In [ ]:
# ── CELDA 5: Helpers de imagen ──────────────────────────────────────────────
def url_absoluta(src, base=DOMINIO):
    """Convierte cualquier src en URL absoluta.
    IMPORTANTE: las imágenes data:image/jpeg;base64,... se conservan tal cual
    porque son la imagen embebida real, no un placeholder."""
    if not src:
        return ''
    src = src.strip()
    # data: URIs son las imágenes reales embebidas → conservar
    if src.startswith('data:'):
        return src
    if src.startswith('http://') or src.startswith('https://'):
        return src
    return urljoin(base, src)

def mejor_imagen(img_tag, base=DOMINIO):
    """
    Lee la URL real de la imagen probando atributos en orden.
    Los temas con lazy-load guardan la URL real en data-src,
    no en src (que suele ser un placeholder gris de 1px).
    En este sitio las imágenes son data:image/jpeg;base64,... embebidas en src.
    """
    if img_tag is None:
        return ''
    for attr in ('data-src', 'data-lazy-src', 'data-original', 'src'):
        url = url_absoluta(img_tag.get(attr, ''), base)
        if url:
            return url
    # Último recurso: srcset
    srcset = img_tag.get('srcset', '')
    if srcset:
        primera = srcset.split(',')[0].strip().split(' ')[0]
        return url_absoluta(primera, base)
    return ''

print('✅ Helpers de imagen OK')


In [ ]:
# ── CELDA 6: Parser de producto (basado en el original) ─────────────────────
def parsear_producto(html):
    """
    Misma lógica que el original (article.product-card + clases exactas),
    con imagen mejorada (data-src antes que src) y URL absoluta.
    """
    soup = BeautifulSoup(html, 'lxml')

    # Marca
    t = soup.find('div', class_='product-card__brand')
    marca = t.text.strip() if t else ''

    # Nombre
    t = soup.find('h3', class_='product-card__title')
    nombre = t.text.strip() if t else ''

    # Precio
    t = soup.find('span', class_='product-card__price')
    precio = t.text.strip() if t else ''

    # Tag (Nuevo, Oferta, etc.)
    t = soup.find('span', class_='product-card__tag')
    tag = t.text.strip() if t else ''

    # Link detalle
    a = soup.find('a')
    link = url_absoluta(a['href']) if a and a.get('href') else ''

    # Imagen — primero data-src, luego src
    img = soup.find('img', class_='product-card__img')
    if img is None:
        img = soup.find('img')  # fallback: cualquier imagen
    imagen = mejor_imagen(img)

    return {
        'marca':  marca,
        'nombre': nombre,
        'precio': precio,
        'tag':    tag,
        'link':   link,
        'imagen': imagen,
    }

print('✅ Parser OK')

In [ ]:
# ── CELDA 7: Paginación real + extracción de HTML ───────────────────────────
# El sitio usa botones button.pagination-btn (páginas 1,2,3...N)
# NO tiene scroll infinito. Hay que hacer click en cada página.

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, StaleElementReferenceException

SELECTOR          = 'article.product-card'
SELECTOR_BTN_PAG  = 'button.pagination-btn'   # botones de página
SELECTOR_BTN_NEXT = 'button.pagination-btn:last-child'  # botón siguiente →

print('🔄 Abriendo navegador...')
driver = crear_driver()
driver.get(URL_BASE)
time.sleep(5)

htmls        = []
pagina_actual = 1

while True:
    # Scroll para activar lazy-load de la página actual
    driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
    time.sleep(1.5)
    driver.execute_script('window.scrollTo(0, 0);')
    time.sleep(1)
    driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
    time.sleep(SCROLL_PAUSA)

    # Extraer productos de esta página
    elementos = driver.find_elements(By.CSS_SELECTOR, SELECTOR)
    nuevos    = []
    for el in elementos:
        try:
            nuevos.append(el.get_attribute('outerHTML'))
        except Exception as e:
            print(f'  ⚠️ No se pudo leer elemento: {e}')

    print(f'  📄 Página {pagina_actual}: {len(nuevos)} productos extraídos')
    htmls.extend(nuevos)

    # Buscar botón siguiente (→) que no esté deshabilitado
    try:
        botones = driver.find_elements(By.CSS_SELECTOR, SELECTOR_BTN_PAG)
        # El último botón es el "→" (siguiente)
        btn_next = botones[-1] if botones else None

        if btn_next is None:
            print('  ⚠️ No se encontraron botones de paginación')
            break

        # Si el botón siguiente está deshabilitado, ya estamos en la última página
        if btn_next.get_attribute('disabled') is not None or 'disabled' in btn_next.get_attribute('class'):
            print(f'  ✅ Última página alcanzada (página {pagina_actual})')
            break

        # Click en siguiente
        driver.execute_script('arguments[0].scrollIntoView(true);', btn_next)
        time.sleep(0.5)
        driver.execute_script('arguments[0].click();', btn_next)  # JS click evita intercepts
        pagina_actual += 1
        time.sleep(SCROLL_PAUSA)  # esperar que cargue la nueva página

    except StaleElementReferenceException:
        print(f'  ⚠️ StaleElement en página {pagina_actual}, reintentando...')
        time.sleep(2)
        continue
    except Exception as e:
        print(f'  ❌ Error de paginación: {e}')
        break

driver.quit()
print(f'\n✅ Total páginas recorridas : {pagina_actual}')
print(f'✅ Total productos extraídos: {len(htmls)}')


In [ ]:
# ── CELDA 8: Procesamiento de 5 en 5 con reintentos ────────────────────────
print(f'🔄 Procesando {len(htmls)} productos en lotes de {BATCH_SIZE}...')
print('=' * 65)

productos_ok    = []
productos_error = []
total_lotes     = math.ceil(len(htmls) / BATCH_SIZE)

for n_lote in range(total_lotes):
    inicio = n_lote * BATCH_SIZE
    fin    = min(inicio + BATCH_SIZE, len(htmls))

    print(f'\n📦 LOTE {n_lote + 1}/{total_lotes}  '
          f'(productos #{inicio + 1} – #{fin})')
    print('-' * 65)

    for idx_local, html in enumerate(htmls[inicio:fin]):
        idx_global = inicio + idx_local + 1
        ok = False

        for intento in range(1, MAX_REINTENTOS + 1):
            try:
                prod = parsear_producto(html)

                # Validación mínima: al menos nombre o precio
                if not prod['nombre'] and not prod['precio']:
                    raise ValueError('nombre y precio vacíos')

                prod['indice'] = idx_global
                productos_ok.append(prod)
                ok = True

                estado_img = '🖼️ ' if prod['imagen'] else '⚠️ sin img'
                print(f'  [{idx_global:>3}] {estado_img} | '
                      f'{prod["nombre"][:38]:<38} | '
                      f'{prod["precio"]}')
                break

            except Exception as e:
                if intento < MAX_REINTENTOS:
                    print(f'  [{idx_global:>3}] ⏳ intento {intento} fallido, reintentando...')
                    time.sleep(0.5)
                else:
                    productos_error.append({
                        'indice': idx_global,
                        'error': str(e),
                        'html_snippet': html[:300]
                    })
                    print(f'  [{idx_global:>3}] ❌ ERROR: {e}')

print('\n' + '=' * 65)
print(f'✅ Extraídos correctamente : {len(productos_ok)}')
print(f'❌ Con error               : {len(productos_error)}')

if productos_error:
    print('\nProductos con error:')
    for pe in productos_error:
        print(f'  #{pe["indice"]} → {pe["error"]}')

In [ ]:
# ── CELDA 9: Guardar CSV ────────────────────────────────────────────────────
# NOTA sobre imágenes: si son data:image/jpeg;base64,... la columna 'imagen'
# contendrá esa cadena completa. Son válidas para mostrar en HTML (<img src=...>)
# pero hacen el CSV muy pesado. Se agrega columna 'imagen_tipo' para diferenciar.

NOMBRE_CSV = 'productos_bikeclubpedalazo.csv'

if productos_ok:
    df = pd.DataFrame(productos_ok)
    # Orden de columnas
    df = df[['indice', 'nombre', 'marca', 'precio', 'tag', 'imagen', 'link']]

    # Clasificar tipo de imagen
    df['imagen_tipo'] = df['imagen'].apply(
        lambda x: 'base64' if x.startswith('data:') else ('url' if x.startswith('http') else 'vacia')
    )

    df.to_csv(NOMBRE_CSV, index=False, encoding='utf-8-sig')

    n_base64 = (df['imagen_tipo'] == 'base64').sum()
    n_url    = (df['imagen_tipo'] == 'url').sum()
    n_vacia  = (df['imagen_tipo'] == 'vacia').sum()

    print(f'✅ CSV guardado: {NOMBRE_CSV}')
    print(f'   Total filas      : {len(df)}')
    print(f'   Imagen base64    : {n_base64}  (embebidas en el HTML)')
    print(f'   Imagen URL       : {n_url}')
    print(f'   Sin imagen       : {n_vacia}')
    print()
    # Mostrar sin la columna imagen (demasiado larga)
    print(df[['indice','nombre','marca','precio','tag','imagen_tipo','link']].head(10).to_string())
else:
    print('⚠️ No hay productos para exportar.')


In [ ]:
# ── CELDA 10 (OPCIONAL): Auditar imágenes ───────────────────────────────────
# Verifica con un HEAD request si cada URL de imagen responde 200 OK.
# Útil para saber cuáles van a verse rotas en la nueva web ANTES de subir.
# Cambia a False si no lo necesitas.
AUDITAR = True

if AUDITAR and productos_ok:
    import requests
    print('🔍 Auditando URLs de imágenes...\n')
    auditoria = []

    for prod in productos_ok:
        url_img = prod.get('imagen', '')
        if not url_img:
            estado = 'SIN_URL'
        else:
            try:
                r = requests.head(url_img,
                                  headers={'User-Agent': 'Mozilla/5.0'},
                                  timeout=8,
                                  allow_redirects=True)
                estado = f'HTTP_{r.status_code}'
            except Exception as e:
                estado = f'ERROR_{type(e).__name__}'

        icono = '✅' if estado == 'HTTP_200' else '❌'
        print(f'  [{prod["indice"]:>3}] {icono} {estado:<15} {url_img[:65]}')
        auditoria.append({
            'indice': prod['indice'],
            'nombre': prod['nombre'],
            'estado_imagen': estado,
            'url_imagen': url_img,
        })

    df_aud = pd.DataFrame(auditoria)
    df_aud.to_csv('auditoria_imagenes.csv', index=False, encoding='utf-8-sig')

    ok_n  = (df_aud['estado_imagen'] == 'HTTP_200').sum()
    err_n = len(df_aud) - ok_n
    print(f'\n✅ Imágenes OK: {ok_n}  |  ❌ Con problema: {err_n}')
    print('📄 Guardado: auditoria_imagenes.csv')

In [ ]:
# ── CELDA 11: Descargar archivos ────────────────────────────────────────────
from google.colab import files

files.download(NOMBRE_CSV)
if AUDITAR:
    files.download('auditoria_imagenes.csv')

print('⬇️ Descargas iniciadas')